# 04 · Exact nearest neighbors and full retrieval

Before the approximate index, we will build a reference retriever: we will calculate the cosine similarity of the query with **every** fragment and sort all results. This is an exact kNN type semantic search. It has a cost proportional to the number of vectors and their dimension; for a small dataset, it provides a simple point of reference.

In [ ]:
# Only cell with imports and project path.
import hashlib
import sys
from pathlib import Path

import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, section_records

## 1. Cosine on three small vectors

A vector close to the direction of the query receives a high score regardless of scale. A zero-dimensional or zero vector requires special handling; the model's embedding should not return it, but the production code must check for it.

In [ ]:
toy_query = np.array([1., 0.])
toy_docs = np.array([[2., 0.], [1., 1.], [-1., 0.]])
toy_cosines = (toy_docs @ toy_query) / (np.linalg.norm(toy_docs, axis=1) * np.linalg.norm(toy_query))
toy_cosines.tolist()

## 2. Exact index in memory

Each matrix position corresponds exactly to one `record`. `argsort` returns positions in descending order. This is a full scan, without ANN and without generating a response. We normalize both ends for calculations, so the dot product = cosine.

In [ ]:
records = section_records()
matrix = embed_texts([row["text"] for row in records])
assert np.all(np.linalg.norm(matrix, axis=1) > 0)
matrix = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
question = "Why is a percentage forecasting metric undefined with zero demand?"
q = embed_texts([question])[0]
q = q / np.linalg.norm(q)
scores = matrix @ q
top = np.argsort(-scores)[:5]
[{"rank": rank, "id": records[int(i)]["id"], "heading": records[int(i)]["heading"],
  "cosine": round(float(scores[i]), 3)} for rank, i in enumerate(top, 1)]

## 3. Candidate ≠ evidence ≠ answer

Retrieve the exact text by ID and check the version/hash before quoting. The ranking may find a thematically close, but insufficient, fragment. Metadata `access_scope` must restrict candidates **before** the text is revealed. Here we show a fictional educational corpus, without the Acme portal's authorization mechanism. The hash detects changes in the text; it does not, by itself, confirm the truth of the content.

In [ ]:
winner = records[int(top[0])]
assert winner["version"] == 1
assert (ROOT / winner["source_path"]).is_file()
assert hashlib.sha256(winner["text"].encode("utf-8")).hexdigest() == winner["content_hash"]
{"id": winner["id"], "source": winner["source_path"], "sha256": winner["content_hash"],
 "excerpt": winner["text"][:550]}

## 4. Small retrieval evaluation

For the question, we prepared a label for the correct **document**, not just a keyword. `hit@k` answers "is at least one of its fragments in the first k results?". This is a weaker measure than a manual sufficiency assessment, but better than stating "the result looks good." Additionally, check the question about the November billing and the question without an answer in the corpus.

In [ ]:
checks = [
    ("Why is percentage error undefined at zero demand?", "SCI-FORECAST-2011"),
    ("What was November 2011 header billing?", "FIN-YTD-2011"),
    ("What is a receipt allocation?", "FIN-CASH-2011"),
]
evaluation = []
for text, expected_doc in checks:
    vector = embed_texts([text])[0]
    vector /= np.linalg.norm(vector)
    ranked = np.argsort(-(matrix @ vector))[:5]
    evaluation.append({"query": text, "expected": expected_doc,
                       "hit@1": records[int(ranked[0])]["document_id"] == expected_doc,
                       "hit@5": any(records[int(i)]["document_id"] == expected_doc for i in ranked)})
evaluation

Even for questions without an answer, the "closest" fragment is returned. This number is a relative ranking, not a calibrated probability of a correct answer. The rejection threshold must be determined on labeled questions, including those without answers.

In [ ]:
unknown = "What was Acme's net profit after tax in 2011?"
unknown_vector = embed_texts([unknown])[0]
unknown_vector /= np.linalg.norm(unknown_vector)
unknown_rank = np.argsort(-(matrix @ unknown_vector))[:3]
[(records[int(i)]["id"], round(float(matrix[int(i)] @ unknown_vector), 3)) for i in unknown_rank]

### Why cosine?

Cosine is easy to interpret as the similarity of **directions** and fits normalized vectors. It is not universally the best metric: the choice depends on the training of the embedding model, the distribution of data, and the evaluation. With L2 normalization, the ranking of cosine similarity and the dot product is the same; Euclidean distance on unit vectors gives the same ordering, because `||a-b||² = 2-2·cos(a,b)`.

**Exercise:** manually label several questions for which the corpus does not provide an answer, and compare their results with questions supported by a source. Is a single threshold sufficient for both groups?